# MobileNetV2: Inverted Residual Bottlenecks and Depthwise Separable Convolutions
### A Ground-Up PyTorch Implementation for Masters-Level Practitioners

---

> *"The most important thing is not to stop questioning. Curiosity has its own reason for existing."*
> — Albert Einstein

---

## Why This Notebook Exists

In 2017, Google researchers published [MobileNetV1](https://arxiv.org/abs/1704.04861), introducing depthwise separable convolutions as a drop-in replacement for standard convolutions in mobile and edge settings. A year later, [MobileNetV2](https://arxiv.org/abs/1801.04381) went further — it introduced the **Inverted Residual Bottleneck**, a building block that is now one of the most widely deployed architectural primitives in production computer vision.

Yet despite its ubiquity, the design rationale behind MobileNetV2 is often taught at the wrong level of abstraction. Papers reference formulas, `torchvision` ships a black-box module, and tutorials tend to skip over the precise tensor mechanics that make the architecture tick.

This notebook refuses to do that.

We are going to build everything from **raw PyTorch primitives** — `nn.Conv2d`, `nn.BatchNorm2d`, `nn.ReLU6` — and we will trace every tensor transformation with explicit `[B, C, H, W]` annotations. By the end, you will understand not just *what* MobileNetV2 does, but *why* each design decision was made, and what trade-off it encodes.

---

## Roadmap

| Stage | Topic | Key Concept |
|---|---|---|
| **1** | Standard Convolution | Baseline cost & parameter count |
| **2** | Depthwise Separable Convolution | Factorized spatial + channel mixing |
| **3** | Linear Bottlenecks & ReLU6 | Information preservation under quantization |
| **4** | Inverted Residual Block | Expand → Depthwise → Project |
| **5** | Full MobileNetV2 Backbone | Stacking blocks with stride control |
| **6** | FLOP & parameter analysis | Efficiency benchmarking vs. ResNet |

This notebook covers **Stages 1 and 2** in full.

---

## Prerequisites

You should be comfortable with:
- PyTorch `nn.Module` anatomy (forward pass, parameter registration)
- Convolution arithmetic: output size given kernel, stride, padding
- Batch normalization mechanics (running stats, training vs. eval mode)

If convolution arithmetic feels shaky, revisit [CS231n Lecture 5](http://cs231n.stanford.edu/slides/2022/lecture_5.pdf) before proceeding.

---

## Environment Setup

We pin exact versions to ensure reproducibility. The notebook was developed and tested against:

```
torch==2.3.0
torchsummary==1.5.1
matplotlib==3.8.4
numpy==1.26.4
```

We deliberately **do not** import `torchvision.models`. Every layer will be authored from scratch using `torch.nn` primitives only.

In [ ]:
from __future__ import annotations

import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

---

## Part 1 — The Standard Convolution: Our Baseline

Before we can appreciate what MobileNetV2 is doing, we need a precise accounting of what a standard convolution *costs*. Let's be rigorous.

### 1.1 Anatomy of a Conv-BN-ReLU Block

The standard building block in most CNNs is not a bare convolution — it is the **Conv → BatchNorm → Activation** triplet. Each component plays a specific role:

- **`nn.Conv2d`** — Applies learned spatial filters. For input `[B, C_in, H, W]` and `C_out` filters of shape `[K, K]`, the output is `[B, C_out, H', W']`.
- **`nn.BatchNorm2d`** — Normalizes each channel's activations across the batch dimension. This stabilizes training and allows higher learning rates. It introduces two *learnable* affine parameters per channel: scale `γ` and shift `β`.
- **Activation** — Introduces non-linearity. Standard CNNs use ReLU; MobileNetV2 uses **ReLU6** (`min(max(x, 0), 6)`) for numerical stability under fixed-point quantization.

### 1.2 Parameter Count

For a standard convolution with:
- Input channels: $C_{in}$
- Output channels: $C_{out}$  
- Kernel size: $K \times K$
- Bias: disabled (standard practice with BatchNorm)

$$\text{Parameters}_{\text{conv}} = C_{out} \times C_{in} \times K \times K$$

BatchNorm adds $2 \times C_{out}$ learnable parameters ($\gamma$ and $\beta$), plus $2 \times C_{out}$ non-learnable running statistics (mean and variance). For large channel counts this is negligible relative to the convolution kernel.

### 1.3 FLOP Count

For each output spatial location $(h', w')$, computing one output channel requires $C_{in} \times K \times K$ multiply-accumulate (MAC) operations. With $C_{out}$ output channels and $H' \times W'$ output spatial positions:

$$\text{MACs}_{\text{std}} = H' \times W' \times C_{in} \times C_{out} \times K^2$$

This is the cost we will dramatically reduce through factorization.

In [ ]:
class StandardConv(nn.Module):
    """
    Standard Conv2d → BatchNorm2d → ReLU6 block.

    This is our baseline: a single unfactored spatial convolution that mixes
    both *spatial information* (via the K×K kernel) and *channel information*
    (via the C_in → C_out projection) in a single operation.

    Tensor flow
    -----------
    Input  : [B, C_in,  H,  W]
    Conv2d : [B, C_out, H', W']   where H' = floor((H + 2P - K) / S) + 1
    BN     : [B, C_out, H', W']   shape unchanged; normalizes per-channel
    ReLU6  : [B, C_out, H', W']   shape unchanged; clips to [0, 6]

    Parameters
    ----------
    in_channels  : Number of channels in the input feature map.
    out_channels : Number of convolutional filters (output channels).
    kernel_size  : Spatial extent of each filter. Default 3 (3×3).
    stride       : Step size of the sliding window. stride=2 halves H and W.
    padding      : Zero-padding added to both spatial dims. Default 1 for
                   'same' spatial size when stride=1 and kernel=3.
    groups       : Controls filter grouping (1 = standard conv). Exposed
                   here so subclasses can override without reimplementing BN.
    bias         : Disabled — BatchNorm absorbs the bias term; keeping both
                   wastes parameters and makes BN's mean-shift redundant.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        groups: int = 1,
        bias: bool = False,
    ) -> None:
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=groups,
            bias=bias,          # Absorbed by BN — no free lunch in parameters
        )
        self.bn = nn.BatchNorm2d(
            num_features=out_channels,
            eps=1e-5,           # Numerical stability in denominator
            momentum=0.1,       # EMA decay for running stats: stat = (1-m)*stat + m*batch
            affine=True,        # Learnable γ (scale) and β (shift) per channel
            track_running_stats=True,  # Maintain running mean/var for inference
        )
        self.act = nn.ReLU6(inplace=True)  # MobileNet family uses ReLU6 throughout

        self._init_weights()

    def _init_weights(self) -> None:
        """
        Kaiming (He) uniform initialization for conv weights — the standard
        choice for layers followed by ReLU. Derives fan-in from kernel geometry.

        BatchNorm γ is initialized to 1, β to 0: identity transform at init
        so the network starts in a well-conditioned linear regime.
        """
        nn.init.kaiming_uniform_(self.conv.weight, mode="fan_out", nonlinearity="relu")
        nn.init.ones_(self.bn.weight)   # γ = 1
        nn.init.zeros_(self.bn.bias)    # β = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        x : [B, C_in,  H,  W]  — input feature map

        Returns
        -------
        out : [B, C_out, H', W']  — activated output feature map
        """
        x = self.conv(x)   # [B, C_in, H, W]  → [B, C_out, H', W']
        x = self.bn(x)     # [B, C_out, H', W'] → [B, C_out, H', W']  (normalize)
        x = self.act(x)    # [B, C_out, H', W'] → [B, C_out, H', W']  (clip to [0,6])
        return x

    @staticmethod
    def count_params(in_channels: int, out_channels: int, kernel_size: int = 3) -> int:
        """Analytical parameter count (conv weights only, no bias, no BN affine)."""
        return out_channels * in_channels * kernel_size * kernel_size

    @staticmethod
    def count_macs(
        in_channels: int,
        out_channels: int,
        h_out: int,
        w_out: int,
        kernel_size: int = 3,
    ) -> int:
        """Analytical MAC count for a single forward pass."""
        return h_out * w_out * in_channels * out_channels * kernel_size ** 2

### 1.4 Verifying the Implementation

Let's sanity-check our `StandardConv` with a concrete tensor. We'll use a batch of 4 ImageNet-scale feature maps: `[4, 32, 112, 112]` — matching the first conv output of MobileNetV2 when fed a 224×224 input.

In [ ]:
# ── Smoke test ────────────────────────────────────────────────────────────────
# Scenario: first conv block of MobileNetV2 (post-stem, before first bottleneck)
#   Input : [B=4, C_in=32, H=112, W=112]
#   Config: 3×3 kernel, stride=1, padding=1 → output spatial unchanged
# ─────────────────────────────────────────────────────────────────────────────
B, C_in, H, W = 4, 32, 112, 112
C_out = 64
K = 3

std_conv = StandardConv(
    in_channels=C_in,
    out_channels=C_out,
    kernel_size=K,
    stride=1,
    padding=1,
).to(DEVICE)

x_in = torch.randn(B, C_in, H, W, device=DEVICE)
x_out = std_conv(x_in)

# Verify shape: H' = floor((112 + 2*1 - 3) / 1) + 1 = 112
assert x_out.shape == (B, C_out, H, W), f"Unexpected shape: {x_out.shape}"

H_out, W_out = x_out.shape[2], x_out.shape[3]
n_params  = StandardConv.count_params(C_in, C_out, K)
n_macs    = StandardConv.count_macs(C_in, C_out, H_out, W_out, K)

print("StandardConv")
print(f"  Input  : {list(x_in.shape)}")
print(f"  Output : {list(x_out.shape)}")
print(f"  Conv weight params : {n_params:,}")
print(f"  MACs (single fwd)  : {n_macs / 1e6:.2f}M")

# ReLU6 verification: all activations should be in [0, 6]
assert x_out.min() >= 0.0, "ReLU6 lower bound violated"
assert x_out.max() <= 6.0, "ReLU6 upper bound violated"
print(f"  Activation range   : [{x_out.min():.3f}, {x_out.max():.3f}]  ✓")

---

## Part 2 — Depthwise Separable Convolution: The Key Factorization

### 2.1 The Core Insight

A standard convolution does two fundamentally different things in a single operation:

1. **Spatial filtering** — convolves each output channel with a $K \times K$ receptive field across the spatial dimensions.
2. **Channel mixing** — projects from $C_{in}$ input channels to $C_{out}$ output channels.

The key question is: *must these happen simultaneously?*

In 2014, Laurent Sifre (in his PhD thesis, later popularized by [Chollet's Xception paper](https://arxiv.org/abs/1610.02357)) observed that they do not. You can **factorize** a standard convolution into two cheap sequential operations:

| Step | Operation | Role | Cost |
|---|---|---|---|
| **Depthwise (DW)** | `groups=C_in` Conv2d | Spatial filtering per channel | $H' \cdot W' \cdot C_{in} \cdot K^2$ MACs |
| **Pointwise (PW)** | `kernel_size=1` Conv2d | Channel mixing | $H' \cdot W' \cdot C_{in} \cdot C_{out}$ MACs |

Total MACs of DSC:
$$\text{MACs}_{\text{DSC}} = H' W' C_{in} K^2 + H' W' C_{in} C_{out} = H' W' C_{in}(K^2 + C_{out})$$

Compare to the standard conv:
$$\text{MACs}_{\text{std}} = H' W' C_{in} C_{out} K^2$$

The **reduction ratio** is:
$$\frac{\text{MACs}_{\text{DSC}}}{\text{MACs}_{\text{std}}} = \frac{1}{C_{out}} + \frac{1}{K^2}$$

For $K=3$ and $C_{out}=64$: ratio $\approx \frac{1}{9} \approx 8\times$ fewer MACs. This is the efficiency gain that enables MobileNet to run on smartphones.

### 2.2 The Depthwise Convolution

A **depthwise convolution** is a grouped convolution with `groups = in_channels`. Instead of one filter per output channel that sees all input channels, we have *one filter per input channel* that sees only its own channel.

Concretely, for `groups = C`:
- The `C_in` input channels are split into `C` groups of 1 channel each.
- Each group has its own `K × K` filter.
- Requires `C_in == C_out` (or `C_out` must be divisible by `groups`). In the standard DW case, `C_out = C_in`.

This means the DW layer has **zero cross-channel interaction** — spatial and channel mixing are completely decoupled.

### 2.3 The Pointwise Convolution

A **pointwise convolution** is a standard `Conv2d` with `kernel_size=1`. At each spatial location, it computes a linear combination of all $C_{in}$ input channels to produce each of $C_{out}$ output channels. This is exactly a *per-pixel matrix multiplication* — it has no spatial extent and therefore no spatial receptive field.

Together, DW + PW achieves the same expressiveness as a standard conv but at a fraction of the cost.

> **Design note:** MobileNetV1 uses DW+PW+BN+ReLU6 at each stage. MobileNetV2 changes this to expand via PW *first*, then DW, then project via a **linear** PW (no activation). This ordering change is the inverted residual — we'll implement that in Stage 4.

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    """
    Depthwise Separable Convolution: Depthwise → Pointwise.

    Factorizes a standard K×K convolution into two sequential operations:

      Stage 1 — Depthwise:
        A grouped Conv2d with groups=in_channels. Each input channel is
        independently filtered by its own K×K spatial kernel. No cross-channel
        information is exchanged. Output has the same number of channels as
        input (channel count preserved, spatial dims controlled by stride).

      Stage 2 — Pointwise:
        A standard 1×1 Conv2d. No spatial extent — operates only across the
        channel axis. Projects the in_channels-dim feature vector at each
        spatial location to out_channels-dim.

    Tensor flow
    -----------
    Input       : [B, C_in,  H,   W  ]
    ── Depthwise stage ─────────────────────────────────────────────────────
    DW Conv2d   : [B, C_in,  H',  W' ]   groups=C_in; spatial K×K per channel
    DW BN       : [B, C_in,  H',  W' ]   normalize per-channel over batch
    DW ReLU6    : [B, C_in,  H',  W' ]   non-linearity; clips to [0, 6]
    ── Pointwise stage ─────────────────────────────────────────────────────
    PW Conv2d   : [B, C_out, H',  W' ]   kernel=1×1; channel projection
    PW BN       : [B, C_out, H',  W' ]   normalize per-channel over batch
    PW ReLU6    : [B, C_out, H',  W' ]   non-linearity; clips to [0, 6]
    ── Output ──────────────────────────────────────────────────────────────
    Output      : [B, C_out, H',  W' ]

    Parameters
    ----------
    in_channels  : Number of input channels (also the DW group count).
    out_channels : Number of output channels after the pointwise projection.
    stride       : Applied to the depthwise conv. stride=2 halves H and W.
                   The pointwise conv always uses stride=1.
    dw_padding   : Padding for the depthwise conv. Default 1 gives 'same'
                   spatial dims for a 3×3 kernel with stride=1.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        dw_padding: int = 1,
    ) -> None:
        super().__init__()

        # ── Stage 1: Depthwise Convolution ───────────────────────────────────
        # groups=in_channels: each channel owns exactly one 3×3 filter.
        # Parameter cost: C_in × 1 × 3 × 3  (vs. C_in × C_out × 3 × 3 standard)
        self.depthwise = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=in_channels,   # DW always preserves channel count
                kernel_size=3,
                stride=stride,
                padding=dw_padding,
                groups=in_channels,         # ← the critical flag: grouped = depthwise
                bias=False,
            ),
            nn.BatchNorm2d(
                num_features=in_channels,
                eps=1e-5,
                momentum=0.1,
                affine=True,
                track_running_stats=True,
            ),
            nn.ReLU6(inplace=True),
        )

        # ── Stage 2: Pointwise Convolution ───────────────────────────────────
        # kernel_size=1: pure channel mixing, zero spatial extent.
        # Parameter cost: C_out × C_in × 1 × 1
        self.pointwise = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=1,
                stride=1,                   # PW never changes spatial dims
                padding=0,                  # No padding needed for 1×1
                groups=1,                   # Full cross-channel connectivity
                bias=False,
            ),
            nn.BatchNorm2d(
                num_features=out_channels,
                eps=1e-5,
                momentum=0.1,
                affine=True,
                track_running_stats=True,
            ),
            nn.ReLU6(inplace=True),
        )

        self._init_weights()

    def _init_weights(self) -> None:
        """
        Initialize depthwise and pointwise conv weights with Kaiming uniform.
        All BatchNorm affine params follow the γ=1, β=0 convention.
        """
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_uniform_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)   # γ
                nn.init.zeros_(module.bias)    # β

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        x   : [B, C_in,  H,  W]   — input feature map

        Returns
        -------
        out : [B, C_out, H', W']  — depthwise-separable convolution output
        """
        x = self.depthwise(x)    # [B, C_in, H, W]  → [B, C_in,  H', W']
        x = self.pointwise(x)   # [B, C_in, H', W'] → [B, C_out, H', W']
        return x

    @staticmethod
    def count_params(in_channels: int, out_channels: int, kernel_size: int = 3) -> dict:
        """Breakdown of conv-weight parameters for DW and PW stages."""
        dw = in_channels * 1 * kernel_size * kernel_size   # one K×K filter per channel
        pw = out_channels * in_channels * 1 * 1            # C_out × C_in 1×1 filters
        return {"depthwise": dw, "pointwise": pw, "total": dw + pw}

    @staticmethod
    def count_macs(
        in_channels: int,
        out_channels: int,
        h_out: int,
        w_out: int,
        kernel_size: int = 3,
    ) -> dict:
        """Analytical MAC count split by stage."""
        dw = h_out * w_out * in_channels * kernel_size ** 2
        pw = h_out * w_out * in_channels * out_channels
        return {"depthwise": dw, "pointwise": pw, "total": dw + pw}

### 2.4 Verifying the Factorization

Let's run the same tensor through both implementations and compare shapes, parameter counts, and MAC counts. The outputs will **not** be numerically equal (they have different learned weights), but the shapes must match and the efficiency numbers should confirm the theoretical $\approx 8\times$ reduction.

In [ ]:
# ── Shape and efficiency comparison ──────────────────────────────────────────
# Same dimensions as the StandardConv smoke test.
B, C_in, H, W = 4, 32, 112, 112
C_out = 64
K = 3

dsc = DepthwiseSeparableConv(
    in_channels=C_in,
    out_channels=C_out,
    stride=1,
    dw_padding=1,
).to(DEVICE)

x_in = torch.randn(B, C_in, H, W, device=DEVICE)
x_out_dsc = dsc(x_in)

assert x_out_dsc.shape == (B, C_out, H, W), f"Shape mismatch: {x_out_dsc.shape}"

H_out, W_out = x_out_dsc.shape[2], x_out_dsc.shape[3]

# ── Parameter comparison ─────────────────────────────────────────────────────
std_params  = StandardConv.count_params(C_in, C_out, K)
dsc_params  = DepthwiseSeparableConv.count_params(C_in, C_out, K)

# ── MAC comparison ───────────────────────────────────────────────────────────
std_macs    = StandardConv.count_macs(C_in, C_out, H_out, W_out, K)
dsc_macs    = DepthwiseSeparableConv.count_macs(C_in, C_out, H_out, W_out, K)

print(f"{'':30s} {'StandardConv':>15s}  {'DSConv':>15s}  {'Ratio':>10s}")
print("-" * 78)
print(f"{'Output shape':30s} {str(list(x_out.shape)):>15s}  {str(list(x_out_dsc.shape)):>15s}")
print(f"{'Conv parameters':30s} {std_params:>15,}  {dsc_params['total']:>15,}  {dsc_params['total']/std_params:>9.3f}x")
print(f"{'  └─ depthwise':30s} {'—':>15s}  {dsc_params['depthwise']:>15,}")
print(f"{'  └─ pointwise':30s} {'—':>15s}  {dsc_params['pointwise']:>15,}")
print(f"{'MACs (single fwd, M)':30s} {std_macs/1e6:>14.2f}M  {dsc_macs['total']/1e6:>14.2f}M  {dsc_macs['total']/std_macs:>9.3f}x")
print(f"{'  └─ depthwise':30s} {'—':>15s}  {dsc_macs['depthwise']/1e6:>13.2f}M")
print(f"{'  └─ pointwise':30s} {'—':>15s}  {dsc_macs['pointwise']/1e6:>13.2f}M")

# Theoretical reduction: 1/C_out + 1/K^2
theoretical_ratio = 1/C_out + 1/K**2
print(f"\nTheoretical ratio: 1/{C_out} + 1/{K}² = {theoretical_ratio:.4f} ({1/theoretical_ratio:.2f}x reduction)")
print(f"Empirical ratio  : {dsc_macs['total']/std_macs:.4f} ({1/(dsc_macs['total']/std_macs):.2f}x reduction)")

# Activation bounds check for DSC output
assert x_out_dsc.min() >= 0.0
assert x_out_dsc.max() <= 6.0
print(f"\nDSC activation range: [{x_out_dsc.min():.3f}, {x_out_dsc.max():.3f}]  ✓")

### 2.5 Inspecting the Depthwise Weights

It is instructive to look at the actual shapes of the weight tensors in each stage. PyTorch stores Conv2d weights in `[C_out, C_in/groups, kH, kW]` format. For the depthwise layer, this means each output channel has `C_in/groups = 1` filter — confirming the per-channel independence.

In [ ]:
# ── Weight tensor shapes ──────────────────────────────────────────────────────
dw_conv = dsc.depthwise[0]   # nn.Conv2d inside the depthwise Sequential
pw_conv = dsc.pointwise[0]   # nn.Conv2d inside the pointwise Sequential

print("Depthwise Conv2d weight shape:")
print(f"  {list(dw_conv.weight.shape)}  — [C_out={C_in}, C_in/groups=1, kH=3, kW=3]")
print(f"  groups = {dw_conv.groups}  (one independent filter per input channel)")

print("\nPointwise Conv2d weight shape:")
print(f"  {list(pw_conv.weight.shape)}  — [C_out={C_out}, C_in={C_in}, kH=1, kW=1]")
print(f"  groups = {pw_conv.groups}  (full cross-channel projection)")

print("\nTotal trainable parameters (including BN affine):")
total_params = sum(p.numel() for p in dsc.parameters() if p.requires_grad)
print(f"  {total_params:,}")

### 2.6 Gradient Flow Sanity Check

Before moving on, let's confirm that gradients flow through both stages cleanly. In a correctly implemented depthwise conv, gradients must be channelwise-independent — the backward pass of a grouped convolution should not mix gradients across groups.

In [ ]:
# ── Gradient flow check ───────────────────────────────────────────────────────
dsc.train()  # Ensure BN uses batch stats, not running stats

x_grad = torch.randn(2, C_in, H, W, device=DEVICE, requires_grad=True)
out_grad = dsc(x_grad)

# Reduce to scalar and backprop
loss = out_grad.sum()
loss.backward()

# Every parameter must have a gradient after backward
for name, param in dsc.named_parameters():
    if param.requires_grad:
        assert param.grad is not None, f"Missing gradient: {name}"
        assert not param.grad.isnan().any(), f"NaN gradient: {name}"

print("Gradient check passed — all parameters received non-NaN gradients.")
print("\nPer-parameter gradient norms:")
for name, param in dsc.named_parameters():
    if param.requires_grad and param.grad is not None:
        print(f"  {name:45s}  grad_norm = {param.grad.norm().item():.4f}")

---

## Summary: What We've Built So Far

| Module | Params (C_in=32, C_out=64, K=3) | MACs @ 112×112 | Reduction |
|---|---|---|---|
| `StandardConv` | 18,432 | 231.2M | 1.0× (baseline) |
| `DepthwiseSeparableConv` | ~2,400 | ~29.0M | ~8× |

The efficiency gain comes entirely from the factorization insight: **spatial filtering and channel mixing are orthogonal operations** that can be performed independently with no loss of representational capacity (in theory), and significant savings in practice.

### What's Next

In **Stage 3**, we will examine why MobileNetV2 switches from ReLU to a **linear bottleneck** in the projection layer. This is a subtle but critical design choice rooted in information theory — applying a non-linearity to a low-dimensional representation can irreversibly destroy information. We will prove this empirically by comparing activation collapse rates and then implement the `LinearBottleneck` accordingly.

In **Stage 4**, we will assemble the `InvertedResidualBlock` — the core of MobileNetV2 — by combining everything built here into the Expand → Depthwise → Project pipeline with a residual shortcut.